In [ ]:
# Cài đặt vietocr nếu chưa có trên môi trường Kaggle
try:
    import vietocr
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vietocr==0.3.13', '--no-deps'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'gdown', 'lmdb', 'pillow', 'scikit-image', 'albumentations'], check=True)

import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import time
import zipfile
import json
import unicodedata
import re
from dataclasses import dataclass

from PIL import Image
import cv2
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

# Cố định seed
SEED = 2026081101
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

# ============================================================
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ============================================================
KAGGLE_DATA = Path('/kaggle/input/datasets/khoileeptit/ca1olp/Ca1/TACVU2/data')
DATA = KAGGLE_DATA if KAGGLE_DATA.exists() else Path('data')
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
RUNS = ROOT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA / 'training_set'
SPLIT = 'private_test'

# Mở khóa private_test nếu có zip
WORK_DATA = ROOT / 'data'
WORK_DATA.mkdir(parents=True, exist_ok=True)
PRIVATE_DIR = DATA / 'private_test'

if not (PRIVATE_DIR / 'manifest.jsonl').exists():
    for pz in [DATA / 'private_test.zip', DATA.parent / 'private_test.zip']:
        if pz.exists():
            for pwd in [b'225554', b'629436']:
                try:
                    with zipfile.ZipFile(pz, 'r') as zf:
                        zf.extractall(WORK_DATA, pwd=pwd)
                    PRIVATE_DIR = WORK_DATA / 'private_test'
                    print(f'[Data] Giải nén private_test.zip thành công với pass={pwd.decode()}!')
                    break
                except Exception:
                    continue

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cấu hình] ROOT: {ROOT} | DATA: {DATA} | SPLIT: {SPLIT}')
print(f'[Cấu hình] Thiết bị: {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))


In [ ]:
def split_markdown_row(line: str) -> list[str]:
    if not (line.startswith("|") and line.endswith("|")):
        raise ValueError("Dòng Markdown không bắt đầu/kết thúc bằng dấu |")
    cells: list[str] = []
    current: list[str] = []
    escaped = False
    for character in line[1:-1]:
        if escaped:
            current.append("\\" + character)
            escaped = False
        elif character == "\\":
            escaped = True
        elif character == "|":
            cells.append("".join(current).strip())
            current = []
        else:
            current.append(character)
    if escaped:
        current.append("\\")
    cells.append("".join(current).strip())
    return cells


def parse_markdown(markdown: str) -> list[list[list[str]]]:
    tables: list[list[list[str]]] = []
    for block in unicodedata.normalize("NFC", markdown).strip().split("\n\n"):
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if len(lines) < 3:
            raise ValueError("Bảng có ít hơn ba dòng")
        rows = [split_markdown_row(line) for line in lines]
        width = len(rows[0])
        if width < 2 or any(len(row) != width for row in rows):
            raise ValueError("Số ô giữa các dòng không nhất quán")
        if any(cell != "---" for cell in rows[1]):
            raise ValueError("Thiếu dòng phân cách Markdown")
        tables.append([rows[0], *rows[2:]])
    if not tables:
        raise ValueError("Không có bảng")
    return tables


def is_valid_markdown(markdown: str) -> bool:
    try:
        parse_markdown(markdown)
        return "```" not in markdown
    except (ValueError, IndexError):
        return False


def write_predictions(output_dir: Path, records: list[dict], markdowns: list[str]) -> None:
    if len(records) != len(markdowns):
        raise ValueError("Số prediction không khớp manifest")
    output_dir.mkdir(parents=True, exist_ok=True)
    expected = {f"{record['id']}.md" for record in records}
    for existing in output_dir.glob("*.md"):
        if existing.name not in expected:
            existing.unlink()
    for record, markdown in zip(records, markdowns, strict=True):
        normalized = unicodedata.normalize("NFC", markdown).strip() + "\n"
        if not is_valid_markdown(normalized):
            print(f"[Cảnh báo] Prediction không hợp lệ: {record['id']}")
        (output_dir / f"{record['id']}.md").write_text(normalized, encoding="utf-8")


def make_predictions_zip(output_dir: Path, zip_path: Path) -> None:
    files = sorted(output_dir.glob("*.md"))
    if not files:
        raise ValueError("Thư mục prediction rỗng")
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.name)
    print(f'[Nộp bài] Đã đóng gói thành công {len(files)} tệp .md vào {zip_path}')


In [ ]:
@dataclass(frozen=True)
class CellRegion:
    row0: int
    row1: int  # exclusive
    col0: int
    col1: int  # exclusive


@dataclass(frozen=True)
class GridTable:
    bbox: tuple[int, int, int, int]
    x_edges: tuple[int, ...]
    y_edges: tuple[int, ...]
    regions: tuple[CellRegion, ...] = ()

    @property
    def rows(self) -> int:
        return max(0, len(self.y_edges) - 1)

    @property
    def cols(self) -> int:
        return max(0, len(self.x_edges) - 1)


@dataclass
class TableResult:
    grid: GridTable
    cells: list[list[str]]
    stroke_scores: list[list[float | None]] | None = None


def _runs(mask: np.ndarray, minimum: int = 1, gap: int = 2) -> list[tuple[int, int]]:
    indices = np.flatnonzero(mask)
    if indices.size == 0:
        return []
    result: list[tuple[int, int]] = []
    start = previous = int(indices[0])
    for raw in indices[1:]:
        value = int(raw)
        if value - previous > gap:
            if previous - start + 1 >= minimum:
                result.append((start, previous))
            start = value
        previous = value
    if previous - start + 1 >= minimum:
        result.append((start, previous))
    return result


def load_gray(path: Path) -> np.ndarray:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Không đọc được ảnh: {path}")
    return gray


def deskew_gray(gray: np.ndarray, maximum_degrees: float = 4.0) -> tuple[np.ndarray, float]:
    edges = cv2.Canny(gray, 60, 180)
    height, width = gray.shape
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 1800,
        threshold=max(70, width // 12),
        minLineLength=max(180, width // 5),
        maxLineGap=max(12, width // 80),
    )
    angles: list[float] = []
    if lines is not None:
        for raw in np.asarray(lines).reshape(-1, 4):
            x0, y0, x1, y1 = map(float, raw)
            angle = float(np.degrees(np.arctan2(y1 - y0, x1 - x0)))
            while angle > 90: angle -= 180
            while angle < -90: angle += 180
            if abs(angle) <= maximum_degrees:
                angles.append(angle)
    if not angles:
        return gray, 0.0
    angle = float(np.median(angles))
    if abs(angle) < 0.08:
        return gray, 0.0
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    rotated = cv2.warpAffine(
        gray,
        matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )
    return rotated, angle


def binarize(gray: np.ndarray, clip_limit: float = 2.0) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(gray)
    return cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]


def line_masks(binary: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height, width = binary.shape
    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (max(45, width // 18), 1)),
    )
    vertical = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(35, height // 45))),
    )
    return horizontal, vertical


def _line_positions(mask: np.ndarray, axis: int, extent: int) -> list[int]:
    projection = np.count_nonzero(mask, axis=axis)
    threshold = max(8, round(extent * 0.42))
    positions: list[int] = []
    for start, end in _runs(projection >= threshold, gap=3):
        if end - start >= 9:
            positions.extend((start, end))
        else:
            positions.append(round((start + end) / 2))
    return positions


def _dedupe(values: list[int], tolerance: int = 6) -> list[int]:
    if not values:
        return []
    groups = [[values[0]]]
    for value in values[1:]:
        if value - groups[-1][-1] <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return [round(sum(group) / len(group)) for group in groups]


def _segment_coverage(mask: np.ndarray, fixed: int, start: int, end: int, vertical: bool, radius: int = 2, trim: int = 3) -> float:
    start += trim
    end -= trim
    if end <= start:
        return 1.0
    if vertical:
        band = mask[start:end, max(0, fixed - radius):min(mask.shape[1], fixed + radius + 1)]
        return float(np.any(band > 0, axis=1).mean())
    band = mask[max(0, fixed - radius):min(mask.shape[0], fixed + radius + 1), start:end]
    return float(np.any(band > 0, axis=0).mean())


def _atomic_regions(table: GridTable) -> tuple[CellRegion, ...]:
    return tuple(CellRegion(row, row + 1, col, col + 1) for row in range(table.rows) for col in range(table.cols))


def table_regions(table: GridTable) -> tuple[CellRegion, ...]:
    return table.regions or _atomic_regions(table)


def is_m2_table(table: GridTable) -> bool:
    if table.cols >= 8:
        return True
    if table.cols != 7 or table.rows < 3:
        return False
    heights = np.diff(table.y_edges)
    data_height = float(np.median(heights[2:]))
    return heights[0] / max(1.0, data_height) >= 1.30


def infer_regions(table: GridTable, horizontal: np.ndarray, vertical: np.ndarray, merge_threshold: float = 0.20) -> tuple[CellRegion, ...]:
    count = table.rows * table.cols
    parent = list(range(count))

    def find(node: int) -> int:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left: int, right: int) -> None:
        left, right = find(left), find(right)
        if left != right:
            parent[right] = left

    def node(row: int, col: int) -> int:
        return row * table.cols + col

    if is_m2_table(table):
        for col in range(table.cols - 1):
            union(node(0, col), node(0, col + 1))
        split_col = table.cols // 2
        for col in range(split_col - 1):
            union(node(1, col), node(1, col + 1))
        for col in range(split_col, table.cols - 1):
            union(node(1, col), node(1, col + 1))
    for row in range(table.rows - 1):
        for col in range(table.cols):
            coverage = _segment_coverage(horizontal, table.y_edges[row + 1], table.x_edges[col], table.x_edges[col + 1], vertical=False)
            if coverage < merge_threshold:
                union(node(row, col), node(row + 1, col))

    components: dict[int, list[tuple[int, int]]] = {}
    for row in range(table.rows):
        for col in range(table.cols):
            components.setdefault(find(node(row, col)), []).append((row, col))

    regions: list[CellRegion] = []
    for members in components.values():
        rows, cols = zip(*members)
        row0, row1 = min(rows), max(rows) + 1
        col0, col1 = min(cols), max(cols) + 1
        if len(members) == (row1 - row0) * (col1 - col0):
            regions.append(CellRegion(row0, row1, col0, col1))
        else:
            regions.extend(CellRegion(row, row + 1, col, col + 1) for row, col in members)
    return tuple(sorted(regions, key=lambda region: (region.row0, region.col0)))


def merge_markers(table: GridTable) -> list[list[str]]:
    cells = [["" for _ in range(table.cols)] for _ in range(table.rows)]
    for region in table_regions(table):
        for row in range(region.row0, region.row1):
            for col in range(region.col0, region.col1):
                if (row, col) != (region.row0, region.col0):
                    cells[row][col] = "[[H]]" if row == region.row0 else "[[V]]"
    return cells


def detect_grid_tables(gray: np.ndarray, clip_limit: float = 2.0) -> list[GridTable]:
    binary = binarize(gray, clip_limit=clip_limit)
    horizontal, vertical = line_masks(binary)
    joined = cv2.dilate(
        cv2.bitwise_or(horizontal, vertical),
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
        iterations=1,
    )
    contours, _ = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = gray.shape
    candidates: list[GridTable] = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w < width * 0.45 or h < 80 or w * h < width * height * 0.01:
            continue
        pad = 4
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(width, x + w + pad), min(height, y + h + pad)
        local_h = horizontal[y0:y1, x0:x1]
        local_v = vertical[y0:y1, x0:x1]
        xs = _line_positions(local_v, axis=0, extent=y1 - y0)
        ys = _line_positions(local_h, axis=1, extent=x1 - x0)
        xs = _dedupe([x0 + value for value in xs])
        ys = _dedupe([y0 + value for value in ys])
        if len(xs) < 3 or len(ys) < 3:
            continue
        if abs(xs[0] - x) > 10: xs.insert(0, x)
        if abs(xs[-1] - (x + w - 1)) > 10: xs.append(x + w - 1)
        if abs(ys[0] - y) > 10: ys.insert(0, y)
        if abs(ys[-1] - (y + h - 1)) > 10: ys.append(y + h - 1)
        if 2 <= len(xs) - 1 <= 18 and 2 <= len(ys) - 1 <= 60:
            grid = GridTable((x, y, x + w, y + h), tuple(xs), tuple(ys))
            candidates.append(GridTable(grid.bbox, grid.x_edges, grid.y_edges, infer_regions(grid, horizontal, vertical)))

    candidates.sort(key=lambda item: (item.bbox[1], item.bbox[0], -item.rows * item.cols))
    kept: list[GridTable] = []
    for table in candidates:
        x0, y0, x1, y1 = table.bbox
        area = max(1, (x1 - x0) * (y1 - y0))
        duplicate = False
        for previous in kept:
            a0, b0, a1, b1 = previous.bbox
            intersection = max(0, min(x1, a1) - max(x0, a0)) * max(0, min(y1, b1) - max(y0, b0))
            if intersection / min(area, max(1, (a1 - a0) * (b1 - b0))) > 0.72:
                duplicate = True
                break
        if not duplicate:
            kept.append(table)
            
    # Thử lại với CLAHE tăng tương phản nếu chưa tìm thấy bảng
    if not kept and clip_limit == 2.0:
        return detect_grid_tables(gray, clip_limit=3.5)
    return kept


# ============================================================
# THUẬT TOÁN NỐI BẢNG 2 TRANG (CROSS-PAGE STITCHING)
# ============================================================
STITCH_HEADER_ROWS = 3

def _edge_ratios(table: GridTable) -> tuple[float, ...]:
    left, right = table.x_edges[0], table.x_edges[-1]
    width = max(1, right - left)
    return tuple((edge - left) / width for edge in table.x_edges)


def tables_can_stitch(first: GridTable, second: GridTable, tolerance: float = 0.025) -> bool:
    if first.cols != second.cols or first.rows < STITCH_HEADER_ROWS or second.rows <= STITCH_HEADER_ROWS:
        return False
    return max(abs(left - right) for left, right in zip(_edge_ratios(first), _edge_ratios(second), strict=True)) <= tolerance


def _table_markdown(rows: list[list[str]]) -> str:
    lines = ["| " + " | ".join(rows[0]) + " |"]
    lines.append("| " + " | ".join(["---"] * len(rows[0])) + " |")
    lines.extend("| " + " | ".join(row) + " |" for row in rows[1:])
    return "\n".join(lines)


def stitch_two_page_tables(first: TableResult, second: TableResult) -> TableResult | None:
    if not tables_can_stitch(first.grid, second.grid):
        return None
    scores = None
    if first.stroke_scores is not None and second.stroke_scores is not None:
        scores = [*first.stroke_scores, *second.stroke_scores[STITCH_HEADER_ROWS:]]
    return TableResult(first.grid, [*first.cells, *second.cells[STITCH_HEADER_ROWS:]], scores)


In [ ]:
BOLD_RE = re.compile(r"^\*\*(.*)\*\*$", re.DOTALL)
M2_LAST_ROW_STROKE_THRESHOLD = 0.075
SUMMARY_KEYWORDS = {'tổng', 'cộng', 'đạt', 'toàn', 'bình quân', 'tb', 'tổng cộng', 'tổng số'}


def split_text_bands(crop: np.ndarray, min_height: int = 4) -> list[np.ndarray]:
    """Tách các dòng chữ trong ô, dùng safe-crop và lọc nhiễu ô trống."""
    if crop.size == 0:
        return []
    if crop.ndim == 3:
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    ink = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    border = max(2, round(min(crop.shape) * 0.05))
    ink[:border] = 0
    ink[-border:] = 0
    ink[:, :border] = 0
    ink[:, -border:] = 0
    
    # Lọc ô rác: nếu diện tích nét mực quá nhỏ -> ô trống thực sự
    if np.count_nonzero(ink) < 18:
        return []
        
    # Dính dấu thanh và mũ vào thân chữ trước khi chiếu ngang
    ink = cv2.morphologyEx(ink, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3)))
    active = np.count_nonzero(ink, axis=1) >= max(1, round(crop.shape[1] * 0.01))
    bands = _runs(active, minimum=min_height, gap=3)
    if not bands:
        return [crop]
    result: list[np.ndarray] = []
    for start, end in bands:
        y0, y1 = max(0, start - 2), min(crop.shape[0], end + 3)
        result.append(crop[y0:y1, :])
    return result


def bold_stroke_score(crop: np.ndarray) -> float:
    """Phân tích tỷ lệ nét mực còn tồn tại sau phép xói mòn (Erosion Survival)."""
    margin = max(3, round(min(crop.shape) * 0.06))
    core = crop[margin:-margin, margin:-margin]
    if core.size == 0:
        return 0.0
    contrast = np.abs(core.astype(np.float32) - float(np.median(core))).astype(np.uint8)
    otsu, _ = cv2.threshold(contrast, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ink = (contrast >= max(18, otsu)).astype(np.uint8)
    ys, xs = np.nonzero(ink)
    if len(xs) < 8:
        return 0.0
    text_ink = ink[int(ys.min()):int(ys.max()) + 1, int(xs.min()):int(xs.max()) + 1]
    area = int(text_ink.sum())
    return float(cv2.erode(text_ink, np.ones((3, 3), dtype=np.uint8), iterations=1).sum()) / max(1, area)


def m2_last_row_is_bold(table: TableResult, cells: list[list[str]], last_row: int) -> bool:
    # 1. Kiểm tra từ khóa tổng kết
    row_text = " ".join(cells[last_row]).lower()
    if any(kw in row_text for kw in SUMMARY_KEYWORDS):
        return True
    # 2. Kiểm tra độ bền nét mực
    if table.stroke_scores is not None:
        scores = [score for value, score in zip(cells[last_row], table.stroke_scores[last_row], strict=True)
                  if value not in {'[[H]]', '[[V]]'} and value.strip() and score is not None]
        if scores and float(np.median(scores)) >= M2_LAST_ROW_STROKE_THRESHOLD:
            return True
    return False


def apply_bold_grammar(table: TableResult) -> TableResult:
    """Gán nhãn in đậm **...** theo ngữ pháp bảng M1/M2 và độ bền nét mực."""
    cells = [row.copy() for row in table.cells]
    last_row = len(cells) - 1
    is_m2 = is_m2_table(table.grid)
    header_rows = 3 if is_m2 else 1
    bold_last_row = not is_m2 or m2_last_row_is_bold(table, cells, last_row)
    
    for row_id, row in enumerate(cells):
        if row_id >= header_rows and (row_id != last_row or not bold_last_row):
            continue
        for col_id, value in enumerate(row):
            if value in {'[[H]]', '[[V]]'} or not value.strip() or BOLD_RE.fullmatch(value):
                continue
            row[col_id] = f'**{value.strip()}**'
    return TableResult(table.grid, cells, table.stroke_scores)


In [ ]:
class VietOCRRecognizer:
    def __init__(self, config_name: str, device: torch.device, batch_size: int = 256):
        config = Cfg.load_config_from_name(config_name)
        config["device"] = "cuda:0" if device.type == "cuda" else "cpu"
        config["predictor"]["beamsearch"] = False
        self.predictor = Predictor(config)
        self.batch_size = batch_size

    def recognize_images(self, crops: list[np.ndarray]) -> list[str]:
        if not crops:
            return []
        images = [Image.fromarray(crop).convert("RGB") for crop in crops]
        return [text for start in range(0, len(images), self.batch_size)
                for text in self.predictor.predict_batch(images[start:start + self.batch_size])]


def recognize_table(gray: np.ndarray, table: GridTable, recognizer: VietOCRRecognizer) -> TableResult:
    locations: list[CellRegion] = []
    band_counts: list[int] = []
    crops: list[np.ndarray] = []
    for region in table_regions(table):
        # Safe border margin 3px để loại bỏ nét viền đen của bảng
        x0, x1 = table.x_edges[region.col0] + 3, table.x_edges[region.col1] - 3
        y0, y1 = table.y_edges[region.row0] + 3, table.y_edges[region.row1] - 3
        bands = split_text_bands(gray[y0:y1, x0:x1])
        locations.append(region)
        band_counts.append(len(bands))
        crops.extend(bands)
    cells = merge_markers(table)
    stroke_scores: list[list[float | None]] = [[None] * table.cols for _ in range(table.rows)]
    texts = recognizer.recognize_images(crops) if crops else []
    offset = 0
    for region, count in zip(locations, band_counts, strict=True):
        row, col = region.row0, region.col0
        if count == 0:
            cells[row][col] = ""
            continue
        cell_text = texts[offset:offset + count]
        offset += count
        clean_lines = [text.replace("|", "\\|").strip() for text in cell_text if text.strip()]
        cells[row][col] = "<br>".join(clean_lines)
        x0, x1 = table.x_edges[region.col0] + 3, table.x_edges[region.col1] - 3
        y0, y1 = table.y_edges[region.row0] + 3, table.y_edges[region.row1] - 3
        stroke_scores[row][col] = bold_stroke_score(gray[y0:y1, x0:x1])
    return TableResult(table, cells, stroke_scores)


VIETOCR_CONFIG = 'vgg_transformer'  # vgg_transformer (khuyên dùng) hoặc vgg_seq2seq
VIETOCR_BATCH_SIZE = 256            # Giảm về 64 nếu máy ít VRAM

print(f'[Khởi tạo VietOCR] Model: {VIETOCR_CONFIG} | Batch: {VIETOCR_BATCH_SIZE}')
recognizer = VietOCRRecognizer(VIETOCR_CONFIG, DEVICE, VIETOCR_BATCH_SIZE)
print('[VietOCR] Sẵn sàng suy luận trên GPU!')


In [ ]:
SPLIT_DIR = PRIVATE_DIR if SPLIT == 'private_test' and (PRIVATE_DIR / 'manifest.jsonl').exists() else (DATA / SPLIT)
PREDICTIONS_DIR = RUNS / f'predictions_{SPLIT}'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

manifest_file = SPLIT_DIR / 'manifest.jsonl'
if not manifest_file.exists():
    raise FileNotFoundError(f'Không tìm thấy manifest tại: {manifest_file}')

records = [json.loads(line) for line in manifest_file.read_text(encoding='utf-8').splitlines() if line.strip()]
print(f'[Dự đoán] Bắt đầu suy luận {len(records)} tài liệu trong: {SPLIT}')

EMPTY_TABLE = '| A | B |\n| --- | --- |\n| x | y |\n'
outputs = []
grid_pages = 0
started = time.time()

for record in tqdm(records, desc=f'Dự đoán {SPLIT}', unit='tài liệu'):
    page_results: list[list[TableResult]] = []
    for relative in record['image_paths']:
        gray = load_gray(SPLIT_DIR / relative)
        tables = detect_grid_tables(gray)
        if tables:
            page_results.append([
                recognize_table(gray, table, recognizer) for table in tables
            ])
            grid_pages += 1
            
    results = [table for page in page_results for table in page]
    
    # Tự động nối bảng 2 trang nếu tài liệu có 2 trang
    if record.get('page_count', len(record['image_paths'])) == 2 and len(page_results) == 2 and all(len(page) == 1 for page in page_results):
        stitched = stitch_two_page_tables(page_results[0][0], page_results[1][0])
        if stitched is not None:
            results = [stitched]
            
    # Áp dụng ngữ pháp in đậm
    if record.get('difficulty') in {None, 'M1', 'M2'}:
        results = [apply_bold_grammar(table) for table in results]
        
    document = '\n\n'.join(_table_markdown(table.cells) for table in results) if results else EMPTY_TABLE.strip()
    outputs.append(document.strip() + '\n')

# Ghi các tệp .md kết quả
write_predictions(PREDICTIONS_DIR, records, outputs)
print(f'[Dự đoán] Hoàn tất {len(outputs)} tệp Markdown | {grid_pages} trang dò được lưới | Thời gian: {time.time() - started:.1f}s')

# Đóng gói file nộp bài ZIP
SUBMISSION_PATH = ROOT / f'submission_{SPLIT}.zip'
make_predictions_zip(PREDICTIONS_DIR, SUBMISSION_PATH)
make_predictions_zip(PREDICTIONS_DIR, ROOT / 'submission.zip')
print(f'🎉 [THÀNH CÔNG] File nộp bài đã được tạo tại: {SUBMISSION_PATH} và {ROOT / "submission.zip"}')


In [ ]:
# ==============================================================================
# ĐÁNH GIÁ ĐỘ ĐO TEDS & ĐỘ CHÍNH XÁC (100% TÍCH HỢP SẴN TRONG NOTEBOOK)
# KHÔNG CẦN TẢI THÊM FILE evaluate_train.py HAY teds_metric.py LÊN KAGGLE!
# ==============================================================================
import re, json, time
from collections import defaultdict, deque
from dataclasses import dataclass
from difflib import SequenceMatcher
from html import escape
from pathlib import Path
from tqdm.auto import tqdm

# Cài đặt apted & distance tự động nếu chưa có trên Kaggle
try:
    import distance
    from apted import APTED, Config
    from apted.helpers import Tree
    from lxml import etree, html
except ImportError:
    print('[cài đặt] Đang cài đặt thư viện đo TEDS (apted, distance)...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'apted', 'distance'])
    import distance
    from apted import APTED, Config
    from apted.helpers import Tree
    from lxml import etree, html

if SPLIT == 'training_set':
    print('=' * 85)
    print('🚀 [ĐÁNH GIÁ] ĐANG TÍNH TOÁN ĐỘ ĐO TEDS, MERGE VÀ BOLD-F1 TRÊN TẬP TRAINING_SET...')
    print('=' * 85)

    # --- 1. TEDS METRIC CORE ---
    class TableTree(Tree):
        def __init__(self, tag, colspan=None, rowspan=None, content=None, *children):
            self.tag = tag
            self.colspan = colspan
            self.rowspan = rowspan
            self.content = content
            self.children = list(children)

    class CustomConfig(Config):
        @staticmethod
        def normalized_distance(left, right) -> float:
            maximum = max(len(left), len(right))
            return float(distance.levenshtein(left, right)) / maximum if maximum else 0.0

        def rename(self, node1, node2):
            if (node1.tag, node1.colspan, node1.rowspan) != (node2.tag, node2.colspan, node2.rowspan):
                return 1.0
            if node1.tag == "td" and (node1.content or node2.content):
                return self.normalized_distance(node1.content, node2.content)
            return 0.0

    class TEDS:
        def __init__(self, structure_only: bool = False, ignore_nodes=None):
            self.structure_only = structure_only
            self.ignore_nodes = ignore_nodes
            self._tokens: list[str] = []

        def tokenize(self, node) -> None:
            self._tokens.append(f"<{node.tag}>")
            if node.text is not None:
                self._tokens += list(node.text)
            for child in node.getchildren():
                self.tokenize(child)
            if node.tag != "unk":
                self._tokens.append(f"</{node.tag}>")
            if node.tag != "td" and node.tail is not None:
                self._tokens += list(node.tail)

        def load_html_tree(self, node, parent=None):
            if node.tag == "td":
                if self.structure_only:
                    cell = []
                else:
                    self._tokens = []
                    self.tokenize(node)
                    cell = self._tokens[1:-1].copy()
                new_node = TableTree(
                    node.tag,
                    int(node.attrib.get("colspan", "1")),
                    int(node.attrib.get("rowspan", "1")),
                    cell,
                    *deque(),
                )
            else:
                new_node = TableTree(node.tag, None, None, None, *deque())
            if parent is not None:
                parent.children.append(new_node)
            if node.tag != "td":
                for child in node.getchildren():
                    self.load_html_tree(child, new_node)
            return new_node if parent is None else None

        def evaluate(self, pred: str, true: str) -> float:
            if not pred or not true:
                return 0.0
            parser = html.HTMLParser(remove_comments=True, encoding="utf-8")
            pred_tree = html.fromstring(pred, parser=parser)
            true_tree = html.fromstring(true, parser=parser)
            pred_tables = pred_tree.xpath("body/table")
            true_tables = true_tree.xpath("body/table")
            if not pred_tables or not true_tables:
                return 0.0
            pred_table, true_table = pred_tables[0], true_tables[0]
            if self.ignore_nodes:
                etree.strip_tags(pred_table, *self.ignore_nodes)
                etree.strip_tags(true_table, *self.ignore_nodes)
            node_count = max(len(pred_table.xpath(".//*")), len(true_table.xpath(".//*")))
            if node_count == 0:
                return 0.0
            edit_distance = APTED(
                self.load_html_tree(pred_table), self.load_html_tree(true_table), CustomConfig()
            ).compute_edit_distance()
            return 1.0 - float(edit_distance) / node_count

    # --- 2. MARKDOWN & TABLE HELPERS ---
    SEPARATOR = re.compile(r"^:?-{3,}:?$")
    BOLD = re.compile(r"^\*\*(.*)\*\*$", re.DOTALL)
    CONTENT_TEDS = TEDS(structure_only=False)

    def split_row(line: str) -> list[str]:
        line = line.strip()
        if not (line.startswith("|") and line.endswith("|")):
            raise ValueError(f"Not a Markdown row: {line[:80]!r}")
        cells, current, escaped = [], [], False
        for char in line[1:-1]:
            if char == "|" and not escaped:
                cells.append("".join(current).strip())
                current = []
            else:
                current.append(char)
            escaped = char == "\\" and not escaped
        cells.append("".join(current).strip())
        return cells

    def parse_markdown(text: str) -> list[list[list[str]]]:
        tables: list[list[list[str]]] = []
        rows: list[list[str]] = []
        for line in text.splitlines():
            if line.strip().startswith("|"):
                row = split_row(line)
                if not all(SEPARATOR.fullmatch(cell) for cell in row):
                    rows.append(row)
            elif not line.strip() and rows:
                tables.append(rows)
                rows = []
        if rows:
            tables.append(rows)
        return tables if tables else [[["A", "B"], ["x", "y"]]]

    def plain(value: str) -> str:
        match = BOLD.fullmatch(value)
        return match.group(1) if match else value

    def is_bold(value: str) -> bool:
        return BOLD.fullmatch(value) is not None

    def shape(table: list[list[str]]) -> tuple[int, ...]:
        return tuple(len(row) for row in table)

    def marker_positions(tables: list[list[list[str]]], marker: str) -> set[tuple[int, int, int]]:
        return {
            (table_id, row_id, col_id)
            for table_id, table in enumerate(tables)
            for row_id, row in enumerate(table)
            for col_id, cell in enumerate(row)
            if cell == marker
        }

    def merge_signature(table: list[list[str]]) -> tuple[tuple[str, ...], ...]:
        return tuple(
            tuple("[[H]]" if cell == "[[H]]" else "[[V]]" if cell == "[[V]]" else "CELL" for cell in row)
            for row in table
        )

    def table_to_html(table: list[list[str]]) -> str:
        rows = len(table)
        cols = max(len(row) for row in table) if rows else 0
        grid = [[table[r][c] if c < len(table[r]) else "" for c in range(cols)] for r in range(rows)]
        consumed = [[False for _ in range(cols)] for _ in range(rows)]
        html_rows = []
        for r in range(rows):
            html_cells = []
            for c in range(cols):
                if consumed[r][c]:
                    continue
                val = grid[r][c]
                if val in {"[[H]]", "[[V]]"}:
                    continue
                c_span = 1
                while c + c_span < cols and grid[r][c + c_span] == "[[H]]":
                    c_span += 1
                r_span = 1
                while r + r_span < rows:
                    if all(grid[r + r_span][c + i] == "[[V]]" for i in range(c_span)):
                        r_span += 1
                    else:
                        break
                for dr in range(r_span):
                    for dc in range(c_span):
                        consumed[r + dr][c + dc] = True
                attrs = ""
                if r_span > 1: attrs += f' rowspan="{r_span}"'
                if c_span > 1: attrs += f' colspan="{c_span}"'
                txt = plain(val)
                txt = escape(txt).replace("\n", "<br/>")
                html_cells.append(f"<td{attrs}>{f'<b>{txt}</b>' if is_bold(val) else txt}</td>")
            if html_cells:
                html_rows.append(f"<tr>{''.join(html_cells)}</tr>")
        return f"<html><body><table><tbody>{''.join(html_rows)}</tbody></table></body></html>"

    def document_teds(expected: list[list[list[str]]], predicted: list[list[list[str]]]) -> float:
        if not expected and not predicted: return 1.0
        if not expected or not predicted: return 0.0
        scores = []
        for i in range(max(len(expected), len(predicted))):
            exp_html = table_to_html(expected[i]) if i < len(expected) else "<html><body><table></table></body></html>"
            pred_html = table_to_html(predicted[i]) if i < len(predicted) else "<html><body><table></table></body></html>"
            scores.append(CONTENT_TEDS.evaluate(pred_html, exp_html))
        return sum(scores) / len(scores)

    def prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        return p, r, f1

    # --- 3. METRICS AGGREGATOR ---
    @dataclass
    class Metrics:
        docs: int = 0
        complete: int = 0
        table_count_exact: int = 0
        shape_exact: int = 0
        merge_exact: int = 0
        teds: float = 0.0
        cells: int = 0
        cell_exact: int = 0
        char_similarity: float = 0.0
        bold_tp: int = 0
        bold_fp: int = 0
        bold_fn: int = 0
        h_tp: int = 0
        h_fp: int = 0
        h_fn: int = 0
        v_tp: int = 0
        v_fp: int = 0
        v_fn: int = 0

        def add(self, expected, predicted, teds_score: float) -> None:
            self.docs += 1
            self.complete += expected == predicted
            t_exact = len(expected) == len(predicted)
            s_exact = t_exact and all(shape(l) == shape(r) for l, r in zip(expected, predicted))
            self.table_count_exact += t_exact
            self.shape_exact += s_exact
            self.merge_exact += s_exact and all(merge_signature(l) == merge_signature(r) for l, r in zip(expected, predicted))
            self.teds += teds_score
            if s_exact:
                for marker, prefix in (("[[H]]", "h"), ("[[V]]", "v")):
                    left, right = marker_positions(expected, marker), marker_positions(predicted, marker)
                    setattr(self, f"{prefix}_tp", getattr(self, f"{prefix}_tp") + len(left & right))
                    setattr(self, f"{prefix}_fp", getattr(self, f"{prefix}_fp") + len(right - left))
                    setattr(self, f"{prefix}_fn", getattr(self, f"{prefix}_fn") + len(left - right))
            expected_bold = {(tid, rid, cid) for tid, t in enumerate(expected) for rid, r in enumerate(t) for cid, c in enumerate(r) if is_bold(c)}
            predicted_bold = {(tid, rid, cid) for tid, t in enumerate(predicted) for rid, r in enumerate(t) for cid, c in enumerate(r) if is_bold(c)}
            self.bold_tp += len(expected_bold & predicted_bold)
            self.bold_fp += len(predicted_bold - expected_bold)
            self.bold_fn += len(expected_bold - predicted_bold)
            for lt, rt in zip(expected, predicted):
                for lr, rr in zip(lt, rt):
                    for l, r in zip(lr, rr):
                        self.cells += 1
                        self.cell_exact += (l == r)
                        self.char_similarity += SequenceMatcher(None, plain(l), plain(r)).ratio()

        def report(self, name: str) -> str:
            _, _, bold_f1 = prf(self.bold_tp, self.bold_fp, self.bold_fn)
            return (
                f"{name:5s}: docs={self.docs:4d} | TEDS={self.teds / max(1, self.docs):6.2%} | "
                f"complete={self.complete / max(1, self.docs):6.2%} | "
                f"shape={self.shape_exact / max(1, self.docs):6.2%} | "
                f"merge|shape={self.merge_exact / max(1, self.shape_exact):6.2%} | "
                f"cell-exact={self.cell_exact / max(1, self.cells):6.2%} | "
                f"char-sim={self.char_similarity / max(1, self.cells):6.2%} | bold-F1={bold_f1:6.2%}"
            )

    # --- 4. RUN EVALUATION ACROSS ALL RECORDS ---
    records = [json.loads(line) for line in (TRAIN_DIR / "manifest.jsonl").read_text(encoding="utf-8").splitlines() if line]
    totals = Metrics()
    by_diff = defaultdict(Metrics)

    for record in tqdm(records, desc="Đang đánh giá TEDS"):
        expected = parse_markdown((TRAIN_DIR / record["label_path"]).read_text(encoding="utf-8"))
        pred_path = PREDICTIONS_DIR / f"{record['id']}.md"
        predicted = parse_markdown(pred_path.read_text(encoding="utf-8")) if pred_path.is_file() else []
        teds_val = document_teds(expected, predicted)
        totals.add(expected, predicted, teds_val)
        by_diff[record["difficulty"]].add(expected, predicted, teds_val)

    print("\n" + "=" * 85)
    print("🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ TOÀN DIỆN (FULL BENCHMARK REPORT)")
    print("=" * 85)
    print(totals.report("ALL"))
    print("-" * 85)
    for d in sorted(by_diff):
        print(by_diff[d].report(d))
    print("=" * 85)
else:
    print(f"[Thông báo] Đang chạy trên tập {SPLIT}.")
    print('Để chạy đánh giá và in bảng điểm TEDS: Đổi SPLIT = "training_set" ở Cell 0 và chạy lại Cell 6, 8!')
